# 21 – ACS Attribute Cleaning (2019–2023, Tracts)

This notebook prepares **ACS attributes** at the census tract level.

**Goals:**
- Load ACS 2019–2023 table(s).
- Identify key variables (population, income, etc.).
- Clean GEOID and ensure it matches tract boundary IDs.
- Subset to the study area if needed.
- Save a cleaned ACS table ready to join to tract geometries.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CENSUS_DIR = DATA_DIR / "census"

CENSUS_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census')

## Raw ACS data source

Update the path below to your ACS table, e.g.:

- `data/census/acs_2019_2023_tracts.csv`

We will save a cleaned table as:

- `data/census/acs_2019_2023_tracts_clean.csv`

In [2]:
raw_acs_path = CENSUS_DIR / "raw/nhgis0005_ds267_20235_tract_E.csv"
acs_clean_path = CENSUS_DIR / "processed/acs_2019_2023_tracts_clean.csv"

raw_acs_path, acs_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/raw/nhgis0005_ds267_20235_tract_E.csv'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/processed/acs_2019_2023_tracts_clean.csv'))

In [3]:
acs = pd.read_csv(raw_acs_path)

print("Rows:", len(acs))
print("Columns:", len(acs.columns))
acs.head()

Rows: 1766
Columns: 218


,GISJOIN,YEAR,STUSAB,REGIONA,DIVISIONA,STATE,STATEA,COUNTY,COUNTYA,COUSUBA,...,ASVHE002,ASVHE003,ASVHE004,ASVHE005,ASVHE006,ASVHE007,ASVHE008,ASVHE009,ASVHE010,ASVHE011
0,GIS Join Match Code,Data File Year,State Postal Abbreviation,Region Code,Division Code,State Name,State Code,County Name,County Code,County Subdivision Code,...,Less than 10.0 percent,10.0 to 14.9 percent,15.0 to 19.9 percent,20.0 to 24.9 percent,25.0 to 29.9 percent,30.0 to 34.9 percent,35.0 to 39.9 percent,40.0 to 49.9 percent,50.0 percent or more,Not computed
1,G0400010942600,2019-2023,AZ,NaN,NaN,Arizona,04,Apache County,001,NaN,...,0,0,0,0,0,0,0,0,0,26
2,G0400010942700,2019-2023,AZ,NaN,NaN,Arizona,04,Apache County,001,NaN,...,73,65,56,14,27,9,0,15,37,73
3,G0400010944000,2019-2023,AZ,NaN,NaN,Arizona,04,Apache County,001,NaN,...,217,66,26,79,20,35,2,3,28,26
4,G0400010944100,2019-2023,AZ,NaN,NaN,Arizona,04,Apache County,001,NaN,...,65,58,0,11,15,15,15,0,35,81


In [4]:
acs.columns

Index(['GISJOIN', 'YEAR', 'STUSAB', 'REGIONA', 'DIVISIONA', 'STATE', 'STATEA',
       'COUNTY', 'COUNTYA', 'COUSUBA',
       ...
       'ASVHE002', 'ASVHE003', 'ASVHE004', 'ASVHE005', 'ASVHE006', 'ASVHE007',
       'ASVHE008', 'ASVHE009', 'ASVHE010', 'ASVHE011'],
      dtype='object', length=218)

## Variable selection

Identify which ACS variables you need for the project, for example:

- Total population
- Median household income
- % renter-occupied housing
- % workers using transit, etc.

Update the next cell with the actual column names.

Also identify the **tract GEOID** column used for joining (e.g., `GEOID`, `GEOID_TRACT`, etc.).

In [5]:
TRACT_GEOID_FIELD = "GISJOIN"

key_vars = [
    TRACT_GEOID_FIELD,
    
    # --- Population base ---
    "ASNQE001",  # Total population (Sex by Age: total)

    # --- Income ---
    "ASQPE001",  # Median household income (2023 inflation-adjusted $)

    # --- Rent ---
    "ASVBE001",  # Median gross rent (dollars)

    # --- Housing units & tenure (for % renters, households per acre) ---
    "ASS8E001",  # Total housing units
    "ASS8E002",  # Occupied housing units (households)
    "ASS8E003",  # Vacant housing units
    "ASS9E001",  # Total occupied housing units (same universe as ASS8E002)
    "ASS9E002",  # Owner-occupied
    "ASS9E003",  # Renter-occupied

    # --- Transportation to work (mode share) ---
    "ASORE001",  # Total workers 16+ (denominator)
    "ASORE003",  # Drove alone
    "ASORE004",  # Carpooled
    "ASORE010",  # Public transportation (excluding taxicab, all modes)
    "ASORE018",  # Bicycle
    "ASORE019",  # Walked
    "ASORE021",  # Worked from home
    "ASORE002",  # Car, truck, or van (all auto commuters)
]
    
acs_subset = acs[key_vars].copy()
acs_subset.head()

,GISJOIN,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E001,ASS9E002,ASS9E003,ASORE001,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002
0,GIS Join Match Code,Total,Median household income in the past 12 months ...,Median gross rent,Total,Occupied,Vacant,Total,Owner occupied,Renter occupied,Total,"Car, truck, or van: Drove alone","Car, truck, or van: Carpooled",Public transportation (excluding taxicab),Bicycle,Walked,Worked from home,"Car, truck, or van"
1,G0400010942600,1429,28661,-666666666,558,441,117,441,415,26,346,249,40,12,0,0,40,289
2,G0400010942700,4857,32448,533,2019,1480,539,1480,1111,369,1275,980,60,0,11,116,108,1040
3,G0400010944000,5685,48154,742,2060,1565,495,1565,1063,502,1736,1514,54,0,0,75,93,1568
4,G0400010944100,6026,29688,593,2136,1488,648,1488,1193,295,1529,1074,50,12,0,236,157,1124


In [6]:
acs_subset[TRACT_GEOID_FIELD] = (
    acs_subset[TRACT_GEOID_FIELD]
    .astype(str)
    .str.strip()
)

acs_subset[TRACT_GEOID_FIELD].head()

0    GIS Join Match Code
1         G0400010942600
2         G0400010942700
3         G0400010944000
4         G0400010944100
Name: GISJOIN, dtype: object

In [7]:
acs_subset.isna().sum()

GISJOIN     0
ASNQE001    0
ASQPE001    0
ASVBE001    0
ASS8E001    0
ASS8E002    0
ASS8E003    0
ASS9E001    0
ASS9E002    0
ASS9E003    0
ASORE001    0
ASORE003    0
ASORE004    0
ASORE010    0
ASORE018    0
ASORE019    0
ASORE021    0
ASORE002    0
dtype: int64

In [8]:
# drop tracts with missing GEOID (just in case)
acs_subset = acs_subset[~acs_subset[TRACT_GEOID_FIELD].isna()].copy()
len(acs_subset)

1766

In [9]:
acs_subset.to_csv(acs_clean_path, index=False)
print("Saved cleaned ACS attributes to:", acs_clean_path)

Saved cleaned ACS attributes to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\census\processed\acs_2019_2023_tracts_clean.csv
